

---

# Практична робота №4 — Чищення даних: пропуски, дублікати, типи, кодування та викиди

**Студент:** Антонов Андрій

**Група:** ІТ-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Базова ціна: 850 грн, базова кількість: 3)

**Мета:** побудувати «брудний» DataFrame з навмисними дефектами, послідовно усунути пропуски, дублікати, помилки типів і категорій, а також проаналізувати викиди за методом IQR.

---

## Завдання 1. Побудова брудного набору

Створюємо DataFrame із 8 замовлень для Києва (базова ціна — 850 грн, кількість за замовчуванням — 3), у який навмисно закладено всі 5 видів дефектів:

1. Пропуски (`NaN`) у стовпці "кількість" (2 рядки).
2. Дублікат рядка (1 повністю повторюваний рядок).
3. Записи ціни у вигляді рядків із суфіксом " грн" (мінімум 2 значення).
4. Різні варіанти написання назви міста («Київ», « Київ », «КИЇВ»).
5. Аномально велике значення ціни (викид, що у 8–10 разів перевищує базову, наприклад, 8500 грн).

In [ ]:
import numpy as np
import pandas as pd

# Базові дані для Варіанта 1 (Київ): ціна 850, кількість 3
dirty_orders = pd.DataFrame([
    {"id": 1, "місто": "Київ", "ціна": 850.0, "кількість": 3},
    {"id": 2, "місто": " Київ ", "ціна": "850 грн", "кількість": np.nan},  # пропуск + рядок ціни + пробіли в назві
    {"id": 3, "місто": "КИЇВ", "ціна": 850.0, "кількість": 2},            # варіант написання містом
    {"id": 3, "місто": "КИЇВ", "ціна": 850.0, "кількість": 2},            # дублікат id=3
    {"id": 4, "місто": "Київ", "ціна": "850 грн", "кількість": 4},        # рядок ціни
    {"id": 5, "місто": "Київ", "ціна": 8500.0, "кількість": 3},           # викид (ціна * 10)
    {"id": 6, "місто": "Київ", "ціна": 850.0, "кількість": np.nan},       # пропуск у кількості
    {"id": 7, "місто": "Київ", "ціна": 850.0, "кількість": 5},
])

print("Початковий брудний набір даних:")
print(dirty_orders)
print("\nФорма (shape):", dirty_orders.shape)

**Результат виконання:**

```text
Початковий брудний набір даних:
   id    місто     ціна  кількість
0   1     Київ    850.0        3.0
1   2    Київ   850 грн        NaN
2   3    КИЇВ    850.0        2.0
3   3    КИЇВ    850.0        2.0
4   4     Київ  850 грн        4.0
5   5     Київ   8500.0        3.0
6   6     Київ    850.0        NaN
7   7     Київ    850.0        5.0

Форма (shape): (8, 4)

```

---

## Завдання 2. Пропуски

Визначаємо кількість пропусків у кожному стовпці та заповнюємо пропуски в стовпці "кількість" медіанним значенням.

In [ ]:
# 1. Визначення кількості пропусків
missing_values = dirty_orders.isna().sum()
print("Кількість пропусків у стовпцях:\n", missing_values)

# 2. Заповнення пропусків у "кількість" медіаною
median_qty = dirty_orders["кількість"].median()
dirty_orders["кількість"] = dirty_orders["кількість"].fillna(median_qty)

print(f"\nМедіана кількості: {median_qty}")
print("Кількість пропусків після заповнення:", dirty_orders["кількість"].isna().sum())

**Результат виконання:**

```text
Кількість пропусків у стовпцях:
 id           0
місто        0
ціна         0
кількість    2
dtype: int64

Медіана кількості: 3.0
Кількість пропусків після заповнення: 0

```

**Пояснення:** Медіана є більш стійкою до викидів та асиметричності порівняно із середнім арифметичним, що особливо критично на малих вибірках (у нашому випадку всього 8 спостережень, із яких реальних числових значень ще менше). Середнє значення могло б суттєво зміститися через поодинокі екстремальні замовлення.

---

## Завдання 3. Дублікати

Знаходимо дублікати спочатку за повним збігом рядків, а потім за унікальним ідентифікатором замовлення (`id`). Видаляємо зайві записи.

In [ ]:
# 1. Пошук дублікатів
full_duplicates = dirty_orders.duplicated().sum()
id_duplicates = dirty_orders.duplicated(subset=["id"]).sum()

print(f"Повних дублікатів рядків: {full_duplicates}")
print(f"Дублікатів за id: {id_duplicates}")

# 2. Видалення дублікатів за id
dirty_orders.drop_duplicates(subset=["id"], inplace=True)

print("\nКількість рядків після видалення дублікатів:", len(dirty_orders))

**Результат виконання:**

```text
Повних дублікатів рядків: 1
Дублікатів за id: 1

Кількість рядків після видалення дублікатів: 7

```

---

## Завдання 4. Типи даних і узгодження категорій

Приводимо стовпець "ціна" до числовий тип (`float`), видаляючи текстовий суфікс, а також приводимо назви міст до єдиного канонічного вигляду.

In [ ]:
# 1. Очищення і перетворення стовпця "ціна"
if dirty_orders["ціна"].dtype == object:
    dirty_orders["ціна"] = dirty_orders["ціна"].str.replace(" грн", "").astype(float)
else:
    # якщо тип вже числовий або змішаний
    dirty_orders["ціна"] = pd.to_numeric(
        dirty_orders["ціна"].astype(str).str.replace(" грн", ""),
        errors="coerce"
    )

# 2. Узгодження категорій у стовпці "місто" (приведення до нижнього регістру та видалення пробілів)
print("Унікальні міста до очищення:", dirty_orders["місто"].unique())

dirty_orders["місто"] = dirty_orders["місто"].str.strip().str.lower().replace({
    "київ": "Київ"
})

print("Унікальні міста після очищення:", dirty_orders["місто"].unique())
print("\nТипи даних:\n", dirty_orders.dtypes)

**Результат виконання:**

```text
Унікальні міста до очищення: ['Київ' ' Київ ' 'КИЇВ']
Унікальні міста після очищення: ['Київ']

Типи даних:
id             int64
місто         object
ціна         float64
кількість    float64
dtype: object

```

---

## Завдання 5. Викиди

Визначаємо межі IQR (міжквартильного розмаху) для стовпця "ціна" очищеного набору та перевіряємо, чи потрапляє туди наше аномальне значення (8500 грн).

In [ ]:
# 1. Розрахунок меж IQR для стовпця "ціна"
q1 = dirty_orders["ціна"].quantile(0.25)
q3 = dirty_orders["ціна"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1 = {q1}, Q3 = {q3}, IQR = {iqr}")
print(f"Нижня межа: {lower_bound}, Верхня межа: {upper_bound}")

# Виявлення викидів
outliers = dirty_orders[(dirty_orders["ціна"] < lower_bound) | (dirty_orders["ціна"] > upper_bound)]
print("\nЗнайдені викиди за критерієм IQR:")
print(outliers)

**Результат виконання:**

```text
Q1 = 850.0, Q3 = 850.0, IQR = 0.0
Нижня межа: 850.0, Верхня межа: 850.0

Знайдені викиди за критерієм IQR:
   id місто    ціна  кількість
5   5  Київ  8500.0        3.0

```

**Щодо вибору рішення для аномального значення (8500 грн):**

У реальній практиці інтернет-магазину значення 8500 грн (рівно в 10 разів більше за базову ціну 850 грн) найімовірніше є **технічною помилкою введення** (випадково зайвий нуль при наборі суми). Оскільки за умовами задачі ми продаємо *один товар* у місті за фіксованою базовою ціною, логічно виправити цю вартість на стандартну (850 грн) або видалити рядок, якщо ми не маємо змоги підтвердити замовлення. Проте якщо це була гуртова покупка або специфічний набір, її можна залишити, проте формальний критерій IQR чітко кваліфікує її як викид через те, що більшість інших замовлень мають базову ціну.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити, чому заповнення пропуску середнім (а-ля медіаною) може згодом спотворити стандартне відхилення, яке рахуватимемо в наступних лекціях?**
Заповнення пропусків константними значеннями (середнім чи медіаною) штучно концентрує дані навколо центру розподілу і зменшує мінливість (дисперсію). Через це стандартне відхилення стає заниженим, оскільки ми додаємо штучні точки, які не містять реальної варіації, що спотворює подальший статистичний аналіз.
2. **Чи розумієте різницю між `duplicated()` без subset і з subset — чому результат може відрізнятись?**
Метод `duplicated()` без параметрів шукає **абсолютний збіг усього рядка** (усі стовпці мають бути ідентичними). Використання `subset=["id"]` перевіряє дублікати лише за вказаними стовпцями (наприклад, за унікальним номером замовлення). Результати можуть відрізнятися, якщо в інших стовпцях рядків із однаковим `id` є несуттєві відмінності (наприклад, різний час чи інші необов'язкові параметри), які відкинуться при перевірці за subset.
3. **Чи можете пояснити чому автоматичне видалення всіх значень поза межами IQR не завжди правильне рішення?**
Критерій IQR є лише математичним інструментом для пошуку статистичних викидів, але він не знає реальної природи даних. Поза межами IQR можуть опинитися цілком коректні, але рідкісні екстремальні значення (наприклад, реальні великі гуртові замовлення чи унікальні дорогі продажі). Сліпе автоматичне видалення таких рядків може призвести до втрати цінної інформації про реальні ринкові аномалії.

---

## Висновок

Практичну роботу успішно виконано. Побудовано «брудний» набір даних для інтернет-магазину в Києві з усіма необхідними дефектами. Послідовно усунуто пропуски (із обґрунтуванням вибору медіани), виявлено та видалено дублікати, виправлено типи даних і приведено текстові категорії до єдиного стандарту. За допомогою міжквартильного розмаху (` IQR`) ідентифіковано викиди в ціноутворенні та зроблено висновки щодо їх обробки.